<!-- @format -->

# Adult Income Prediction

## Machine Learning Pipeline — Main Notebook

Notebook này là **entry point** duy nhất để chạy toàn bộ pipeline.

**Cách chạy:** `Runtime → Run all`

Pipeline bao gồm:

1. Setup môi trường
2. Exploratory Data Analysis (EDA)
3. Data Preprocessing
4. Classical ML Pipeline (Feature Extraction → Baseline → Tuning)
5. Deep Learning Pipeline (MLP + TabNet)
6. Tổng kết và so sánh toàn bộ mô hình


<!-- @format -->

## 1. Setup môi trường


In [ ]:
# [Setup 1.1] Cài đặt thư viện bổ sung
!pip install imbalanced-learn optuna pytorch-tabnet -q

In [ ]:
# [Setup 1.2] Thiết lập đường dẫn module
# Ưu tiên 1: tìm modules/ từ cấu trúc thư mục đã giải nén (zip)
# Ưu tiên 2: clone từ GitHub nếu không tìm thấy cục bộ

import os, sys

REPO_URL    = 'https://github.com/Hanne2202/ml-project-group10.git'
REPO_BRANCH = 'main'
REPO_DIR    = '/content/ml-project-group10'

def _find_modules_path():
    """Tìm thư mục gốc chứa modules/ theo nhiều hướng."""
    candidates = [
        os.path.abspath('..'),          # chạy từ notebooks/ (chuẩn khi giải nén zip)
        os.path.abspath('.'),           # chạy từ root repo
        '/content/ml-project-group10', # Colab sau khi clone
    ]
    for path in candidates:
        if os.path.exists(os.path.join(path, 'modules')):
            return path
    return None

MODULES_PATH = _find_modules_path()

if MODULES_PATH is None:
    # Không tìm thấy cục bộ -> clone từ GitHub
    print('[INFO] Không tìm thấy modules/ cục bộ -> Clone từ GitHub...')
    if os.path.exists(REPO_DIR):
        print('     -> Repo đã tồn tại, lấy code mới nhất...')
        os.system('git -C ' + REPO_DIR + ' fetch origin ' + REPO_BRANCH)
        os.system('git -C ' + REPO_DIR + ' reset --hard origin/' + REPO_BRANCH)
    else:
        os.system('git clone -b ' + REPO_BRANCH + ' ' + REPO_URL + ' ' + REPO_DIR)
    MODULES_PATH = _find_modules_path()

if MODULES_PATH is None:
    raise FileNotFoundError('Không tìm thấy modules/. Vui lòng kiểm tra lại cấu trúc thư mục.')

sys.path.insert(0, MODULES_PATH)
print('[OK] modules/ tìm thấy tại:', MODULES_PATH)
print('[Ready] sys.path đã được cập nhật.')


In [ ]:
# [Setup 1.2] Thiết lập đường dẫn module

import os, sys

def _find_modules_path():
    """Tìm thư mục gốc chứa modules/ theo nhiều hướng."""
    candidates = [
        os.path.abspath('..'),           # chạy từ notebooks/ (chuẩn)
        os.path.abspath('.'),            # chạy từ root repo
        '/content/ml-project-group10',  # Colab sau khi upload zip
    ]
    for path in candidates:
        if os.path.exists(os.path.join(path, 'modules')):
            return path
    return None

MODULES_PATH = _find_modules_path()

if MODULES_PATH is None:
    raise FileNotFoundError(
        'Không tìm thấy thư mục modules/. '
        'Hãy đảm bảo giải nén đúng cấu trúc thư mục và '
        'chạy notebook từ bên trong thư mục notebooks/.'
    )

sys.path.insert(0, MODULES_PATH)
print('[OK] modules/ tìm thấy tại:', MODULES_PATH)
print('[Ready] sys.path đã được cập nhật.')


In [ ]:
# [Setup 1.3] Import toàn bộ modules
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import torch

import modules.eda as eda
import modules.preprocessing as prep
import modules.classical_learning as cl
import modules.deep_learning as dl
import modules.tuning as tune

print("[OK] Tất cả modules đã được import thành công.")
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"[OK] Device: {device}")

In [ ]:
# [Setup 1.4] Đường dẫn dữ liệu và thư mục output
# Dữ liệu được tải từ nguồn công khai trên GitHub (không mount Drive cá nhân)
DATA_URL = "https://raw.githubusercontent.com/Hanne2202/ml-group10-data/main/adult.csv"

os.makedirs(os.path.join(MODULES_PATH, 'features'), exist_ok=True)
os.makedirs(os.path.join(MODULES_PATH, 'results'),  exist_ok=True)
os.makedirs(os.path.join(MODULES_PATH, 'models'),   exist_ok=True)

FEATURES_DIR = os.path.join(MODULES_PATH, 'features')
RESULTS_DIR  = os.path.join(MODULES_PATH, 'results')
MODELS_DIR   = os.path.join(MODULES_PATH, 'models')

print(f"Data URL     : {DATA_URL}")
print(f"Features dir : {FEATURES_DIR}")
print(f"Results dir  : {RESULTS_DIR}")
print(f"Models dir   : {MODELS_DIR}")

<!-- @format -->

---

## 2. Exploratory Data Analysis (EDA)

Phân tích tổng quan dữ liệu Adult Income Dataset trước khi xây dựng pipeline.
Bao gồm: cấu trúc dữ liệu, missing value, phân phối target, và các đặc trưng quan trọng.


In [ ]:
# [EDA 2.1] Load dataset từ nguồn công khai
df = eda.load_data(DATA_URL)

In [ ]:
# [EDA 2.2] Tổng quan cấu trúc dataset
eda.dataset_overview(df)
num_cols, cat_cols = eda.get_column_types(df)

In [ ]:
# [EDA 2.3] Kiểm tra missing value
eda.check_missing_values(df)
eda.plot_missing_summary(df)
eda.inspect_categorical_values(df, cat_cols)

In [ ]:
# [EDA 2.4] Phân phối biến mục tiêu
eda.plot_target_distribution(df, target_col='income')

In [ ]:
# [EDA 2.5] Phân tích categorical features theo target
selected_cat_cols = ['education', 'occupation', 'marital-status']
eda.plot_categorical_by_target(df, selected_cat_cols, target_col='income')

In [ ]:
# [EDA 2.6] Phân tích numerical features theo target
selected_num_cols = ['age', 'hours-per-week', 'educational-num']
eda.plot_numerical_by_target(df, selected_num_cols, target_col='income')

In [ ]:
# [EDA 2.7] Phân tích capital-gain và capital-loss
eda.analyze_capital_feature(df, 'capital-gain', target_col='income')
eda.analyze_capital_feature(df, 'capital-loss', target_col='income')

In [ ]:
# [EDA 2.8] Correlation heatmap và kiểm tra redundancy
eda.plot_correlation_heatmap(df, num_cols)
eda.check_education_redundancy(df)

<!-- @format -->

---

## 3. Data Preprocessing

Tiền xử lý dữ liệu cho classical pipeline:

- Chuẩn hóa missing value (`?` → NaN)
- Loại bỏ cột dư thừa (`education`, `fnlwgt`)
- Chia train/test (80/20, stratify)
- So sánh 4 cấu hình preprocessing và chọn cấu hình tốt nhất theo F1-score


In [ ]:
# [Preprocessing 3.1] Làm sạch dữ liệu
df_prep = df.copy()
df_prep = prep.drop_columns(df_prep, ['education', 'fnlwgt'])
df_prep.replace('?', np.nan, inplace=True)

X = df_prep.drop(columns=['income'])
y = df_prep['income']

num_features = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
cat_features = X.select_dtypes(include=['object']).columns.tolist()

print('Numerical features :', num_features)
print('Categorical features:', cat_features)

In [ ]:
# [Preprocessing 3.2] Train/test split (80/20, stratify)
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f'X_train: {X_train.shape}  |  X_test: {X_test.shape}')

In [ ]:
# [Preprocessing 3.3] So sánh 4 cấu hình preprocessing
preprocessing_configs = prep.get_all_classical_configs(num_features, cat_features)

lr_results_df = prep.compare_preprocessing_configs(
    preprocessing_configs, X_train, X_test, y_train, y_test, pos_label='>50K'
)
print(lr_results_df)

In [ ]:
# [Preprocessing 3.4] Chọn cấu hình tốt nhất
best_config_name, best_preprocessor, X_train_processed, X_test_processed = \
    prep.select_best_preprocessor(preprocessing_configs, lr_results_df, X_train, X_test)

print(f'Best config: {best_config_name}')
print(f'X_train_processed: {X_train_processed.shape}')

<!-- @format -->

---

## 4. Classical Machine Learning Pipeline

Xây dựng và so sánh các mô hình học máy truyền thống:

- **Feature extraction**: Full features (91 chiều) và PCA 95%
- **Baseline**: Logistic Regression, Linear SVM, Random Forest, Gaussian Naive Bayes
- **Hyperparameter tuning**: GridSearch / RandomizedSearch với 5-fold Stratified CV
- **Đánh giá**: F1-score, ROC-AUC, Confusion Matrix, Feature Importance


In [ ]:
# [Classical 4.1] Trích xuất đặc trưng: Full features và PCA 95%
feature_sets, y_train_model, y_test_model = cl.extract_features_pca(
    X_train_processed, X_test_processed,
    y_train, y_test,
    pca_variance=0.95
)
cl.plot_pca_variance(feature_sets, pca_key='pca_95')

In [ ]:
# [Classical 4.2] Lưu feature files
cl.save_features(feature_sets, y_train_model, y_test_model, output_dir=FEATURES_DIR)

In [ ]:
# [Classical 4.3] Huấn luyện baseline models
baseline_models = cl.get_baseline_models()
trained_models, baseline_results_df, baseline_predictions = cl.run_baseline(
    baseline_models, feature_sets, y_train_model, y_test_model
)

baseline_results_df

In [ ]:
# [Classical 4.4] Kết quả baseline
best_baseline = baseline_results_df.iloc[0]
cl.print_best_model_summary(baseline_results_df, stage='Baseline')

cl.plot_f1_comparison(
    baseline_results_df,
    score_col='f1_score',
    title='Baseline Model Comparison by F1-score'
)

best_key = f"{best_baseline['feature_set']}__{best_baseline['model']}"
cl.plot_confusion_matrix(
    y_test_model, baseline_predictions[best_key],
    title=f"Confusion Matrix — {best_baseline['model']} ({best_baseline['feature_set']})"
)
cl.print_classification_report(
    y_test_model, baseline_predictions[best_key], title=best_key
)

In [ ]:
# [Classical 4.5] Lưu kết quả baseline
cl.save_results(baseline_results_df, 'baseline_results.csv', output_dir=RESULTS_DIR)

In [ ]:
# [Classical 4.6] Hyperparameter tuning (5-fold Stratified CV)
tuning_results_df, tuned_models, tuned_predictions = cl.run_tuning(
    feature_sets,
    y_train_model,
    y_test_model,
    main_feature_set='full_features',
    n_splits=5,
)

tuning_results_df

In [ ]:
# [Classical 4.7] Kết quả tuning và ROC curves
best_tuned = tuning_results_df.iloc[0]
cl.print_best_model_summary(tuning_results_df, stage='Tuned')

cl.plot_f1_comparison(
    tuning_results_df.rename(columns={'test_f1_score': 'f1_score'}),
    score_col='f1_score',
    title='Tuned Model Comparison by F1-score'
)

X_test_tune = feature_sets['full_features']['X_test']
cl.plot_roc_curves(tuned_models, X_test_tune, y_test_model)

best_tuned_name   = best_tuned['model']
best_tuned_y_pred = tuned_predictions[best_tuned_name]
cl.plot_confusion_matrix(
    y_test_model, best_tuned_y_pred,
    title=f'Confusion Matrix — Tuned {best_tuned_name}'
)
cl.print_classification_report(y_test_model, best_tuned_y_pred, title=f'Tuned {best_tuned_name}')

In [ ]:
# [Classical 4.8] So sánh baseline vs tuned + feature importance
final_comparison_classical = cl.build_final_comparison(best_baseline, best_tuned)
improvement_df = cl.compute_improvement(best_baseline, best_tuned)
cl.plot_metric_comparison(final_comparison_classical)

fi_df = cl.plot_feature_importance(
    tuned_models, best_preprocessor, num_features, cat_features,
    top_n=15, model_key='Random Forest'
)

In [ ]:
# [Classical 4.9] Lưu kết quả và mô hình cuối cùng
cl.save_results(tuning_results_df, 'tuning_results.csv', output_dir=RESULTS_DIR)
cl.save_results(final_comparison_classical, 'final_model_comparison_classical.csv', output_dir=RESULTS_DIR)

classical_model_path = cl.save_model(tuned_models[best_tuned_name], best_tuned_name, output_dir=MODELS_DIR)

<!-- @format -->

---

## 5. Deep Learning Pipeline

Pipeline học sâu để so sánh với classical pipeline:

- **Logistic Regression** làm baseline cho DL
- **MLP** (Multi-Layer Perceptron) với SMOTE cân bằng lớp
- **Optuna tuning** tìm kiếm hyperparameter tự động cho MLP
- **TabNet** — kiến trúc attention-based cho tabular data

> Phần Optuna search (MLP và TabNet) tốn thời gian nên được hardcode từ kết quả chạy trước. Bỏ comment để chạy lại toàn bộ.


In [ ]:
# [DL 5.1] Preprocessing step-by-step cho MLP (có SMOTE)
from modules.preprocessing import (
    drop_columns, drop_missing_values, map_target_variable,
    apply_ohe, scale_numeric, split_data, apply_smote,
)

df_clean = df.copy()
df_clean = drop_columns(df_clean, columns=['education', 'fnlwgt'])
df_clean = drop_missing_values(df_clean)
df_clean = map_target_variable(df_clean, target_col='income')

TARGET_COL = 'income'
dl_num_cols = df_clean.select_dtypes(include=['int64', 'float64']).columns.tolist()
dl_cat_cols = df_clean.select_dtypes(include=['object']).columns.tolist()
if TARGET_COL in dl_num_cols: dl_num_cols.remove(TARGET_COL)
if TARGET_COL in dl_cat_cols: dl_cat_cols.remove(TARGET_COL)

X_train_raw, X_val_raw, X_test_raw, y_train_dl, y_val_dl, y_test_dl = split_data(
    df_clean, target_col=TARGET_COL, test_size=0.2, val_size=0.1, random_state=42
)
X_train_ohe, ohe_encoder = apply_ohe(X_train_raw, dl_cat_cols)
X_val_ohe,  _            = apply_ohe(X_val_raw,   dl_cat_cols, encoder=ohe_encoder)
X_test_ohe, _            = apply_ohe(X_test_raw,  dl_cat_cols, encoder=ohe_encoder)

X_train_sc, scaler = scale_numeric(X_train_ohe, dl_num_cols)
X_val_sc,   _      = scale_numeric(X_val_ohe,   dl_num_cols, scaler=scaler)
X_test_sc,  _      = scale_numeric(X_test_ohe,  dl_num_cols, scaler=scaler)

X_train_sm, y_train_sm = apply_smote(X_train_sc, y_train_dl, sampling_strategy='auto', k_neighbors=5)

print(f'Train (SMOTE): {X_train_sm.shape}  |  Val: {X_val_sc.shape}  |  Test: {X_test_sc.shape}')

In [ ]:
# [DL 5.2] Logistic Regression baseline cho DL
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
    classification_report, confusion_matrix, ConfusionMatrixDisplay
)

lr_dl = LogisticRegression(max_iter=1000, random_state=42, solver='lbfgs')
lr_dl.fit(X_train_sm, y_train_sm)

y_test_pred_lr  = lr_dl.predict(X_test_sc)
y_test_proba_lr = lr_dl.predict_proba(X_test_sc)[:, 1]

print('=== Logistic Regression (DL baseline) — Test Set ===')
print(classification_report(y_test_dl, y_test_pred_lr, target_names=['<=50K', '>50K']))

test_metrics_lr = {
    'accuracy' : round(accuracy_score(y_test_dl,  y_test_pred_lr),  4),
    'precision': round(precision_score(y_test_dl, y_test_pred_lr),  4),
    'recall'   : round(recall_score(y_test_dl,    y_test_pred_lr),  4),
    'f1'       : round(f1_score(y_test_dl,        y_test_pred_lr),  4),
    'roc_auc'  : round(roc_auc_score(y_test_dl,   y_test_proba_lr), 4),
}

In [ ]:
# [DL 5.3] MLP cơ bản [256, 128, 64]
prep_mlp = {
    'X_train': X_train_sm.values.astype('float32'),
    'X_val'  : X_val_sc.values.astype('float32'),
    'X_test' : X_test_sc.values.astype('float32'),
    'y_train': y_train_sm.values.astype('int64'),
    'y_val'  : y_val_dl.values.astype('int64'),
    'y_test' : y_test_dl.values.astype('int64'),
    'input_dim': X_train_sm.shape[1],
}

train_loader, val_loader, test_loader = dl.create_dataloaders(prep_mlp, batch_size=256)

model_mlp = dl.MLP(input_dim=prep_mlp['input_dim'], hidden_dims=[256, 128, 64], dropout=0.4)
history_mlp = dl.train_model(
    model_mlp, train_loader, val_loader,
    epochs=100, lr=1e-3, weight_decay=1e-3, patience=20,
    device=device, use_pos_weight=False
)
dl.plot_learning_curves(history_mlp)
test_metrics_mlp, y_true_mlp, y_pred_mlp, y_proba_mlp = dl.evaluate_model(
    model_mlp, test_loader, device=device
)

In [ ]:
# [DL 5.4] MLP Optuna tuning (mock study — hardcode từ kết quả chạy trước)
# Bỏ comment phần tune.run_optuna_search() để chạy lại toàn bộ (~7-10 phút)

prep_tuned = {
    'X_train': X_train_sc.values.astype('float32'),
    'X_val'  : X_val_sc.values.astype('float32'),
    'X_test' : X_test_sc.values.astype('float32'),
    'y_train': y_train_dl.values.astype('int64'),
    'y_val'  : y_val_dl.values.astype('int64'),
    'y_test' : y_test_dl.values.astype('int64'),
    'input_dim': X_train_sc.shape[1],
}

# study_mlp = tune.run_optuna_search(prep=prep_tuned, n_trials=30, epochs=50, patience=10, device=device)

class _MockStudy:
    def __init__(self, best_params, best_value):
        self.best_params = best_params
        self.best_value  = best_value

study_mlp = _MockStudy(
    best_params={
        'n_layers': 3, 'hidden_dim_0': 320, 'hidden_dim_1': 448, 'hidden_dim_2': 32,
        'dropout': 0.35, 'lr': 0.00015193253929678293,
        'weight_decay': 3.0308702059789768e-05, 'batch_size': 512, 'optimizer': 'AdamW'
    },
    best_value=0.7049062049062049
)
print(f'Best Val F1 (Optuna MLP): {study_mlp.best_value:.4f}')
print(f'Best Params: {study_mlp.best_params}')

best_model_optuna, best_history_optuna, best_metrics_optuna = tune.train_best_optuna_model(
    study=study_mlp, prep=prep_tuned, epochs=100, patience=15, device=device
)
dl.plot_learning_curves(best_history_optuna)

In [ ]:
# [DL 5.5] TabNet preprocessing và training
prep_tabnet = prep.preprocess_for_tabnet(df_clean, target_col='income', random_state=42)

print('\n--- Training TabNet ---')
tabnet_model = dl.train_tabnet_model(
    prep=prep_tabnet,
    n_d=32, n_a=32, n_steps=3, gamma=1.3,
    lr=2e-2, max_epochs=100, patience=15,
    batch_size=512, virtual_batch_size=128, device=device
)
metrics_tabnet, y_true_tabnet, y_pred_tabnet, y_proba_tabnet = dl.evaluate_tabnet(
    tabnet_model, prep_tabnet
)

In [ ]:
# [DL 5.6] TabNet Optuna tuning (mock study — hardcode từ kết quả chạy trước)
# Bỏ comment phần tune.run_optuna_search_tabnet() để chạy lại (~30-40 phút)

# study_tabnet = tune.run_optuna_search_tabnet(
#     prep_tabnet=prep_tabnet, n_trials=10, max_epochs=50, patience=10,
#     device=device, use_pretraining=True
# )

study_tabnet = _MockStudy(
    best_params={
        'width': 8, 'n_steps': 4, 'gamma': 1.0137868611216592,
        'lambda_sparse': 0.003924034647132935, 'lr': 0.007085132090225794,
        'batch_size': 512, 'virtual_batch_size': 256,
        'pretrain_epochs': 50, 'pretraining_ratio': 0.6195014675689636
    },
    best_value=0.6885694729637235
)
print(f'Best Val F1 (Optuna TabNet): {study_tabnet.best_value:.4f}')

best_tabnet_optuna, metrics_tabnet_best = tune.train_best_optuna_tabnet_model(
    study=study_tabnet, prep_tabnet=prep_tabnet,
    max_epochs=100, patience=15, device=device, use_pretraining=True
)
_, y_true_tb, y_pred_tb, y_proba_tb = dl.evaluate_tabnet(best_tabnet_optuna, prep_tabnet)

<!-- @format -->

---

## 6. Tổng kết — So sánh toàn bộ mô hình

Bảng so sánh đầy đủ giữa:

- Classical pipeline: Logistic Regression, Linear SVM, Random Forest, Gaussian Naive Bayes
- Deep Learning pipeline: MLP (baseline), MLP (Optuna), TabNet (baseline), TabNet (Optuna)


In [ ]:
# [Summary 6.1] Bảng so sánh toàn bộ mô hình
_, y_true_optuna, _, y_proba_optuna = dl.evaluate_model(
    best_model_optuna,
    dl.create_dataloaders(prep_tuned, batch_size=256)[2],
    device=device
)

summary_all = pd.DataFrame([
    # Classical
    {'Pipeline': 'Classical', 'Model': best_baseline['model'] + ' (baseline)',
     'Accuracy': best_baseline['accuracy'], 'Precision': best_baseline['precision'],
     'Recall': best_baseline['recall'], 'F1-Score': best_baseline['f1_score'],
     'ROC-AUC': best_baseline['roc_auc']},
    {'Pipeline': 'Classical', 'Model': best_tuned['model'] + ' (tuned)',
     'Accuracy': best_tuned['test_accuracy'], 'Precision': best_tuned['test_precision'],
     'Recall': best_tuned['test_recall'], 'F1-Score': best_tuned['test_f1_score'],
     'ROC-AUC': best_tuned['test_roc_auc']},
    # Deep Learning
    {'Pipeline': 'Deep Learning', 'Model': 'Logistic Regression (DL baseline)',
     'Accuracy': test_metrics_lr['accuracy'], 'Precision': test_metrics_lr['precision'],
     'Recall': test_metrics_lr['recall'], 'F1-Score': test_metrics_lr['f1'],
     'ROC-AUC': test_metrics_lr['roc_auc']},
    {'Pipeline': 'Deep Learning', 'Model': 'MLP [256,128,64]',
     'Accuracy': test_metrics_mlp['accuracy'], 'Precision': test_metrics_mlp['precision'],
     'Recall': test_metrics_mlp['recall'], 'F1-Score': test_metrics_mlp['f1_score'],
     'ROC-AUC': round(roc_auc_score(y_true_mlp, y_proba_mlp), 4)},
    {'Pipeline': 'Deep Learning', 'Model': 'MLP (Optuna Best)',
     'Accuracy': best_metrics_optuna['accuracy'], 'Precision': best_metrics_optuna['precision'],
     'Recall': best_metrics_optuna['recall'], 'F1-Score': best_metrics_optuna['f1_score'],
     'ROC-AUC': round(roc_auc_score(y_true_optuna, y_proba_optuna), 4)},
    {'Pipeline': 'Deep Learning', 'Model': 'TabNet (baseline)',
     'Accuracy': metrics_tabnet['accuracy'], 'Precision': metrics_tabnet['precision'],
     'Recall': metrics_tabnet['recall'], 'F1-Score': metrics_tabnet['f1_score'],
     'ROC-AUC': round(roc_auc_score(y_true_tabnet, y_proba_tabnet), 4)},
    {'Pipeline': 'Deep Learning', 'Model': 'TabNet (Optuna Best)',
     'Accuracy': metrics_tabnet_best['accuracy'], 'Precision': metrics_tabnet_best['precision'],
     'Recall': metrics_tabnet_best['recall'], 'F1-Score': metrics_tabnet_best['f1_score'],
     'ROC-AUC': round(roc_auc_score(y_true_tb, y_proba_tb), 4)},
])

summary_all = summary_all.sort_values('F1-Score', ascending=False).reset_index(drop=True)
print(summary_all.to_string(index=False))

In [ ]:
# [Summary 6.2] Biểu đồ so sánh F1-Score toàn bộ mô hình
import matplotlib.pyplot as plt
import numpy as np

colors = ['steelblue' if p == 'Classical' else 'coral'
          for p in summary_all['Pipeline']]

plt.figure(figsize=(12, 6))
bars = plt.barh(summary_all['Model'], summary_all['F1-Score'], color=colors, alpha=0.85)
plt.xlabel('F1-Score')
plt.ylabel('Model')
plt.title('So sánh F1-Score — Classical vs Deep Learning Pipeline')
plt.gca().invert_yaxis()
plt.axvline(x=summary_all['F1-Score'].max(), linestyle='--', color='green',
            alpha=0.5, label=f"Best: {summary_all['F1-Score'].max():.4f}")
plt.legend()
plt.grid(axis='x', alpha=0.3)

# Legend phân biệt Classical / Deep Learning
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor='steelblue', alpha=0.85, label='Classical'),
                   Patch(facecolor='coral',     alpha=0.85, label='Deep Learning')]
plt.legend(handles=legend_elements, loc='lower right')
plt.tight_layout()
plt.show()

In [ ]:
# [Summary 6.3] Lưu bảng tổng kết
cl.save_results(summary_all, 'final_summary_all_models.csv', output_dir=RESULTS_DIR)

print('\n=== PIPELINE HOÀN THÀNH ===')
print(f'Tất cả kết quả đã được lưu tại: {RESULTS_DIR}')
print(f'Features đã lưu tại            : {FEATURES_DIR}')
print(f'Model cuối cùng lưu tại        : {MODELS_DIR}')

print('\n=== MÔ HÌNH TỐT NHẤT ===')
best_overall = summary_all.iloc[0]
print(f"Pipeline  : {best_overall['Pipeline']}")
print(f"Model     : {best_overall['Model']}")
print(f"F1-Score  : {best_overall['F1-Score']}")
print(f"ROC-AUC   : {best_overall['ROC-AUC']}")
print(f"Accuracy  : {best_overall['Accuracy']}")

<!-- @format -->

---

## Kết luận

Notebook này đã thực hiện đầy đủ pipeline học máy cho bài toán phân loại thu nhập trên Adult Income Dataset.

**Classical Pipeline:**

- So sánh 4 cấu hình preprocessing, chọn `config_2_onehot_constant_standard` theo F1-score
- Huấn luyện và tuning 4 mô hình: Logistic Regression, Linear SVM, Random Forest, Gaussian Naive Bayes
- Trích xuất đặc trưng với Full features (91 chiều) và PCA 95%

**Deep Learning Pipeline (điểm cộng):**

- MLP với SMOTE cân bằng lớp và Optuna hyperparameter tuning
- TabNet với self-supervised pretraining và Optuna tuning

**Nhận xét chung:**
Trên dữ liệu dạng bảng, các mô hình truyền thống (đặc biệt Logistic Regression và Random Forest) cho hiệu năng rất cạnh tranh so với deep learning. MLP và TabNet cải thiện nhẹ ở một số chỉ số, cho thấy deep learning chỉ thực sự phát huy ưu thế khi dữ liệu lớn hơn hoặc có cấu trúc phức tạp hơn.
